# New-Product Extraction from SEC Form 8-K Filings (NER + LLM)

**Course:** CAP5619 AI in Finance  
**GitHub:** https://github.com/Simy0204/sec-8k-product-extraction

This notebook drives `extract_8k_products.py`. Pipeline: S&P 500 universe → SEC EDGAR 8-K filings (Items 7.01/8.01) → press-release exhibits (EX-99.x) → keyword pre-filter → spaCy + rule-based NER → local LLM via Ollama (JSON schema output) → validation → CSV.

## 0. Setup
```bash
pip install -r requirements.txt
python -m spacy download en_core_web_sm
ollama pull llama3.1:8b      # or: ollama pull deepseek-r1:8b
ollama serve                  # if not already running
```

In [1]:
import os, requests
from pathlib import Path
import pandas as pd
import extract_8k_products as ex

# SEC fair-access policy: identify yourself (name + email)
os.environ.setdefault('SEC_USER_AGENT', 'Bingchun Chen bi035264@ucf.edu')

# Check that Ollama is up and which models are installed
print([m['name'] for m in requests.get('http://localhost:11434/api/tags', timeout=5).json()['models']])

['llama3.1:8b']


## 1. NER demo on the assignment example
spaCy's small model finds the organization but often mislabels product names (e.g. as PERSON). The rule-based pattern (capitalised phrase after a launch verb) recovers them, and the LLM makes the final call.

In [2]:
nlp = ex.load_nlp()
s = 'Apple Inc. has announced the launch of its latest MacBook Pro, featuring a new M3 processor.'
print('spaCy entities :', [(e.text, e.label_) for e in nlp(s).ents])
print('rule-based     :', ex.rule_based_products(s))
passage, cands = ex.run_ner(nlp, s)
print('candidates     :', cands)

spaCy entities : [('Apple Inc.', 'ORG'), ('MacBook Pro', 'PERSON'), ('M3', 'GPE')]
rule-based     : ['MacBook Pro']
candidates     : {'ORG': ['Apple Inc.'], 'PRODUCT': ['MacBook Pro']}


## 2. Configure the run
`max_documents` counts filings actually sent to the LLM (those that pass the launch-keyword pre-filter). Re-running resumes where it stopped: filings already in `extraction_audit.csv` are skipped.

In [3]:
cfg = ex.Config(
    user_agent=os.environ['SEC_USER_AGENT'],
    start_date='2025-11-01', end_date='2026-10-31',
    max_documents=100,
    backend='ollama', model='llama3.1:8b',
    tickers=None,            # e.g. ['AAPL','NVDA','MSFT'] for a quick test
    filing_time_format='date',
    out_dir=Path('output'),
)
cfg

Config(user_agent='Bingchun Chen bi035264@ucf.edu', start_date='2025-11-01', end_date='2026-10-31', max_documents=100, max_filings_per_company=15, items_filter=('7.01', '8.01'), tickers=None, backend='ollama', model='llama3.1:8b', ollama_url='http://localhost:11434', openai_url='https://api.openai.com/v1', passage_chars=6000, filing_time_format='date', verify=True, out_dir=WindowsPath('output'), cache_dir=WindowsPath('data/cache'), sp500_csv=WindowsPath('data/sp500.csv'), sec_sleep=0.15)

## 3. Quick single-filing test (optional)

In [ ]:
test_cfg = ex.Config(**{**cfg.__dict__, 'tickers': ['NVDA'], 'max_documents': 2, 'out_dir': Path('output_test')})
ex.run(test_cfg)

## 4. Full run

In [ ]:
%%time
df = ex.run(cfg)
df.head(20)

## 5. Results and checks

In [4]:
df = pd.read_csv(cfg.out_csv)
audit = pd.read_csv(cfg.audit_csv)

print('Filings scanned          :', len(audit))
print(audit['status'].value_counts().to_string())
llm_docs = audit['status'].str.startswith('llm').sum()
hit = (audit['status'] == 'llm_products').sum()
print(f'\nDocuments sent to LLM    : {llm_docs}')
print(f'Documents with a product : {hit} ({hit / max(llm_docs, 1):.0%})')
print('Product rows             :', len(df), '| companies:', df['stock_name'].nunique())
print('Max description length   :', df['product_description'].str.len().max())
assert (df['product_description'].str.len() < 180).all()
rej = audit['rejected'].dropna()
print('Rejected by validation   :', rej.str.count(';').add(1).sum())

Filings scanned          : 250
status
skipped_no_launch_language    216
llm_no_product                 21
llm_products                   13

Documents sent to LLM    : 34
Documents with a product : 13 (38%)
Product rows             : 29 | companies: 5
Max description length   : 116
Rejected by validation   : 57


In [5]:
print(ex.write_pipe_table(cfg.out_csv, cfg.table_txt))

Company Name           | Stock Name | Filing Time | New Product                                                   | Product Description
--------------------------------------------------------------------------------------------------------------------------------------------
Abbott Laboratories    | ABT        | 2026-10-21  | Ensure                                                        | World’s No. 1 adult nutrition supplement drink
Abbott Laboratories    | ABT        | 2026-10-21  | Pedialyte                                                     | Pediatric hydration drink
Abbott Laboratories    | ABT        | 2026-10-21  | High-protein Ensure shake                                     | High-protein Ensure shake with 42 grams of protein
Abbott Laboratories    | ABT        | 2026-10-21  | Dye-free Pedialyte products                                   | Dye-free Pedialyte products
Aflac                  | AFL        | 2026-08-06  | Anshin Palette, the new medical insurance product      

## 6. Manual accuracy check
Sample 20 rows, open each `url` from the audit file and mark whether the product and description are correct. Report the precision in the write-up.

In [6]:
sample = df.sample(min(20, len(df)), random_state=42).merge(
    audit[['ticker', 'filing_date', 'url']].rename(columns={'ticker': 'stock_name', 'filing_date': 'filing_time'}),
    on=['stock_name', 'filing_time'], how='left').drop_duplicates()
sample['correct'] = ''
sample.to_csv('output/manual_review.csv', index=False)
sample

,company_name,stock_name,filing_time,new_product,product_description,url,correct
0,Amazon,AMZN,2026-02-27,Customized models,Customized models for Amazon’s customer-facing...,https://www.sec.gov/Archives/edgar/data/101872...,
1,Advanced Micro Devices,AMD,2026-05-05,Ryzen AI Embedded P100 Series,AI processors for industrial and edge applicat...,https://www.sec.gov/Archives/edgar/data/2488/0...,
2,Advanced Micro Devices,AMD,2026-08-04,AMD Instinct MI350P GPU,Seamless AI acceleration to existing infrastru...,https://www.sec.gov/Archives/edgar/data/2488/0...,
3,Amazon,AMZN,2026-04-09,Amazon Fresh,Mainstream physical grocery stores with grocer...,https://www.sec.gov/Archives/edgar/data/101872...,
4,Aflac,AFL,2025-11-04,Miraito,Aflac's cancer insurance product launched in M...,https://www.sec.gov/Archives/edgar/data/4977/0...,
5,Advanced Micro Devices,AMD,2026-08-04,AMD Helios rackscale solution,World's most powerful AI server rack,https://www.sec.gov/Archives/edgar/data/2488/0...,
6,Amazon,AMZN,2026-04-09,Amazon Now,Ultra-fast delivery service with 20-minute del...,https://www.sec.gov/Archives/edgar/data/101872...,
7,Abbott Laboratories,ABT,2026-10-21,Ensure,World’s No. 1 adult nutrition supplement drink,https://www.sec.gov/Archives/edgar/data/1800/0...,
8,Amazon,AMZN,2026-02-27,Stateful Runtime Environment,Stateful Runtime Environment powered by OpenAI...,https://www.sec.gov/Archives/edgar/data/101872...,
9,Advanced Micro Devices,AMD,2026-08-04,AMD Ryzen 7700X3D processors,Extended support for Socket AM5 platforms,https://www.sec.gov/Archives/edgar/data/2488/0...,
